<a href="https://colab.research.google.com/github/jwe4/makemore_from_scratch/blob/main/build_makemore2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
%matplotlib inline

In [5]:
#read in all the words
words = open('names.txt', 'r').read().splitlines()
words[:8]

['emma', 'olivia', 'ava', 'isabella', 'sophia', 'charlotte', 'mia', 'amelia']

In [6]:
len(words)

32033

In [7]:
# build the vocabulart of characters and mappings to/from integers
chars = sorted(list(set(''.join(words))))
stoi = {s:i+1 for i,s in enumerate(chars)}
stoi['.'] = 0
itos = {i:s for s,i in stoi.items()}
print(itos)

{1: 'a', 2: 'b', 3: 'c', 4: 'd', 5: 'e', 6: 'f', 7: 'g', 8: 'h', 9: 'i', 10: 'j', 11: 'k', 12: 'l', 13: 'm', 14: 'n', 15: 'o', 16: 'p', 17: 'q', 18: 'r', 19: 's', 20: 't', 21: 'u', 22: 'v', 23: 'w', 24: 'x', 25: 'y', 26: 'z', 0: '.'}


In [8]:
# build the dataset
block_size = 3 # context length: how any characters do we take to predict the next one
# X is input, Y is labels

X,Y = [],[]
for w in words[:5]:
  print(w)
  context = [0] * block_size
  for ch in w + '.':
    ix= stoi[ch]
    X.append(context)
    Y.append(ix)
    print(''.join(itos[i] for i in context), '-->', itos[ix])
    context = context[1:] + [ix] # crop and append

X = torch.tensor(X)
Y = torch.tensor(Y)

emma
... --> e
..e --> m
.em --> m
emm --> a
mma --> .
olivia
... --> o
..o --> l
.ol --> i
oli --> v
liv --> i
ivi --> a
via --> .
ava
... --> a
..a --> v
.av --> a
ava --> .
isabella
... --> i
..i --> s
.is --> a
isa --> b
sab --> e
abe --> l
bel --> l
ell --> a
lla --> .
sophia
... --> s
..s --> o
.so --> p
sop --> h
oph --> i
phi --> a
hia --> .


In [9]:
X.shape, X.dtype, Y.shape, Y.dtype

(torch.Size([32, 3]), torch.int64, torch.Size([32]), torch.int64)

In [10]:
X

tensor([[ 0,  0,  0],
        [ 0,  0,  5],
        [ 0,  5, 13],
        [ 5, 13, 13],
        [13, 13,  1],
        [ 0,  0,  0],
        [ 0,  0, 15],
        [ 0, 15, 12],
        [15, 12,  9],
        [12,  9, 22],
        [ 9, 22,  9],
        [22,  9,  1],
        [ 0,  0,  0],
        [ 0,  0,  1],
        [ 0,  1, 22],
        [ 1, 22,  1],
        [ 0,  0,  0],
        [ 0,  0,  9],
        [ 0,  9, 19],
        [ 9, 19,  1],
        [19,  1,  2],
        [ 1,  2,  5],
        [ 2,  5, 12],
        [ 5, 12, 12],
        [12, 12,  1],
        [ 0,  0,  0],
        [ 0,  0, 19],
        [ 0, 19, 15],
        [19, 15, 16],
        [15, 16,  8],
        [16,  8,  9],
        [ 8,  9,  1]])

In [11]:
Y

tensor([ 5, 13, 13,  1,  0, 15, 12,  9, 22,  9,  1,  0,  1, 22,  1,  0,  9, 19,
         1,  2,  5, 12, 12,  1,  0, 19, 15, 16,  8,  9,  1,  0])

In [12]:
# the 27 character embedding
C = torch.randn((27,2))

In [13]:
C

tensor([[-0.3069,  1.0962],
        [ 0.3642,  1.1411],
        [-1.5546, -0.2135],
        [ 0.2851,  1.9172],
        [ 1.3103, -0.3805],
        [ 1.8103,  1.9583],
        [-0.2893, -0.8484],
        [ 0.2794,  2.0286],
        [ 0.4710, -0.7471],
        [-0.2927,  0.2890],
        [ 0.5876,  0.4505],
        [ 1.0066,  1.5567],
        [-1.7230,  0.2010],
        [ 2.0896,  0.3928],
        [-1.4670, -0.1950],
        [-1.8103, -1.0832],
        [ 1.6391, -0.0351],
        [ 0.8374,  0.7119],
        [-0.0107,  0.1541],
        [ 0.2844,  0.4706],
        [-0.9385, -0.1425],
        [ 0.8790, -0.7345],
        [ 0.1842, -0.1352],
        [-0.0591, -1.0518],
        [-0.4334, -0.5821],
        [ 1.6070, -0.0942],
        [ 1.4395, -0.6971]])

In [15]:
# encode an example integer( integers represent the characters)

F.one_hot(torch.tensor(5), num_classes=27)

tensor([0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0])

In [16]:
F.one_hot(torch.tensor(5), num_classes=27).float() @ C

tensor([1.8103, 1.9583])

In [17]:
C[5]

tensor([1.8103, 1.9583])

In [19]:
# will use index rather than one hot
# can index by X
C[X].shape

torch.Size([32, 3, 2])

In [20]:
X[13,2]

tensor(1)

In [21]:
C[X][13,2]
# Yes, the weight matrix W1 for the hidden layer is defined as (6, 100), meaning 6 rows and 100 columns.

tensor([0.3642, 1.1411])

In [22]:
C[1]

tensor([0.3642, 1.1411])

In [23]:
# so have embedding
emb = C[X]
emb.shape

torch.Size([32, 3, 2])

In [ ]:
# now building the hidden layer
W1 = torch.randn(6,100)
b1 = torch.randn(100)

In [25]:
emb[:,0,:].shape

torch.Size([32, 2])

In [ ]:
emb[:,0,:]

In [28]:
torch.cat([emb[:,0,:], emb[:,1,:],emb[:,2,:]],1).shape

torch.Size([32, 6])

In [30]:
# can do something similar with unbind
torch.cat(torch.unbind(emb,1),1).shape

torch.Size([32, 6])

In [31]:
# still another approach
a = torch.arange(18)
a

tensor([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17])

In [32]:
a.shape

torch.Size([18])

In [33]:
a.view(9,2) # view is very efficient

tensor([[ 0,  1],
        [ 2,  3],
        [ 4,  5],
        [ 6,  7],
        [ 8,  9],
        [10, 11],
        [12, 13],
        [14, 15],
        [16, 17]])

In [34]:
emb.shape

torch.Size([32, 3, 2])

In [36]:
emb.view(32,6) == torch.cat(torch.unbind(emb,1),1)
#


tensor([[True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, T

In [38]:
# back to the hidden layer interface
W1 = torch.randn(6,100)
b1 = torch.randn(100)
h = emb.view(32,6) @ W1 + b1
h.shape

torch.Size([32, 100])